# For1Sight — Corrected ETL Pipeline V2

## Purpose
Transform the 2024–2026 TracingInsights `laptimes.json` files into one row per lap, validate the records, prepare a dashboard-compatible Silver dataset, and optionally publish it to local Parquet and HDFS.

## Why V2 is different
The previous notebook repeatedly triggered Spark jobs against the 6,904 small JSON files. V2 removes the unnecessary self-join, avoids full-row `dropDuplicates()`, consolidates quality metrics, and writes one temporary Parquet staging dataset. All subsequent checks read the staging Parquet instead of repeatedly recomputing the raw JSON transformations.

## Safety
- Raw JSON files are read only.
- The existing `hf_laptimes.parquet` is not overwritten.
- Temporary staging is written under the Windows temporary directory (a path without spaces).
- Local and HDFS exports are disabled by default.
- New output destinations use a new name and `errorifexists` protection.

**Execution:** After a kernel restart, run the cells from the beginning in order. Do not run an isolated later cell because Spark DataFrames are held in kernel memory.


## 1. Imports and project configuration

The notebook uses local Python, PySpark and the existing raw data. No internet connection or API download is required.


In [1]:
import glob
import os
import shutil
import tempfile
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, input_file_name, regexp_replace, regexp_extract,
    arrays_zip, explode, size, array_repeat, coalesce,
    lower, upper, trim, when, least, greatest,
    count, countDistinct, sum as F_sum
)
from pyspark.sql.types import ArrayType

# Locate raw_hf_data from the project folder or its parent.
possible_raw_roots = [
    Path.cwd() / "raw_hf_data",
    Path.cwd().parent / "raw_hf_data",
]
RAW_ROOT = next((p.resolve() for p in possible_raw_roots if p.exists()), None)

if RAW_ROOT is None:
    raise FileNotFoundError(
        "Cannot find raw_hf_data. Open this notebook from the For1Sight "
        "project folder or update possible_raw_roots."
    )

PROJECT_ROOT = RAW_ROOT.parent

# Use Windows TEMP, avoiding the project path's spaces for Spark file I/O.
TEMP_ROOT = Path(tempfile.gettempdir()) / "for1sight_etl_v2"
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

STAGING_PATH = TEMP_ROOT / "flat_laps_stage"
STAGING_URI = STAGING_PATH.as_uri()

LOCAL_TEMP_EXPORT_PATH = TEMP_ROOT / "hf_laptimes_clean_export"
LOCAL_TEMP_EXPORT_URI = LOCAL_TEMP_EXPORT_PATH.as_uri()

# Keep local Spark shuffle overhead reasonable for this laptop-sized workload.
spark = (
    SparkSession.builder
    .appName("For1Sight_Corrected_ETL_V2")
    .config("spark.driver.memory", "4g")
    .config("spark.local.dir", str(TEMP_ROOT / "spark_local"))
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.sparkContext.setLogLevel("WARN")

print("Spark session ready")
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Project root:", PROJECT_ROOT)
print("Raw data root:", RAW_ROOT)
print("Temporary staging:", STAGING_URI)


Spark session ready
Spark version: 3.5.3
Spark master: local[*]
Project root: C:\Users\ayepd\Desktop\For1Sight Project
Raw data root: C:\Users\ayepd\Desktop\For1Sight Project\raw_hf_data
Temporary staging: file:///C:/Users/ayepd/AppData/Local/Temp/for1sight_etl_v2/flat_laps_stage


## 2. Discover and read raw lap-time JSON files

This cell discovers all `laptimes.json` files under the three downloaded seasons. It deliberately does not call `.count()` because that would launch a Spark job before the transformation is ready.


In [2]:
laptimes_files = sorted(
    glob.glob(str(RAW_ROOT / "**" / "laptimes.json"), recursive=True)
)

if not laptimes_files:
    raise FileNotFoundError(f"No laptimes.json files found under {RAW_ROOT}")

print(f"Discovered source files: {len(laptimes_files):,}")

lap_df = (
    spark.read
    .option("multiLine", "true")
    .json(laptimes_files)
    .withColumn("source_file", input_file_name())
)

print("JSON schema inferred. No full-data count has been requested.")
lap_df.printSchema()


Discovered source files: 6,904
JSON schema inferred. No full-data count has been requested.
root
 |-- compound: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- dNum: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- del: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- delR: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- drv: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ff1G: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- fresh: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- iacc: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- lSD: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- lST: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- lap: array (nullable = true)
 |    |-- element: double (contains

## 3. Identify array fields and calculate per-file array lengths

The raw JSON stores lap attributes as parallel arrays. Each index represents one lap. We calculate the minimum and maximum non-null array lengths directly on each source record.

**V2 correction:** We do not create a separate audit DataFrame and join it back to `lap_df`. That self-join was unnecessary and added work.


In [15]:

# ==========================================================
# SECTION 3 — NULL-SAFE ARRAY LENGTH VALIDATION
# ==========================================================

from pyspark.sql.functions import (
    col,
    size,
    when,
    lit,
    least,
    greatest
)

from pyspark.sql.types import ArrayType

print("=" * 70)
print("FOR1SIGHT — NULL-SAFE ARRAY VALIDATION")
print("=" * 70)

# ----------------------------------------------------------
# 1. IDENTIFY ARRAY FIELDS FROM GLOBAL JSON SCHEMA
# ----------------------------------------------------------

array_fields = [
    field
    for field in lap_df.schema.fields
    if isinstance(field.dataType, ArrayType)
]

array_names = [
    field.name
    for field in array_fields
]

print("Total global array fields:", len(array_names))

# ----------------------------------------------------------
# 2. CALCULATE ARRAY LENGTHS SAFELY
# ----------------------------------------------------------

array_sizes = [
    when(
        col(field.name).isNotNull(),
        size(col(field.name))
    ).otherwise(
        lit(None).cast("int")
    )
    for field in array_fields
]

# IMPORTANT:
# Missing optional arrays become NULL, not -1.
# Spark least() and greatest() ignore NULL values.

# ----------------------------------------------------------
# 3. CALCULATE MINIMUM AND MAXIMUM ARRAY LENGTH
# ----------------------------------------------------------

lap_with_lengths = (
    lap_df
    .withColumn(
        "min_array_length",
        least(*array_sizes)
    )
    .withColumn(
        "max_array_length",
        greatest(*array_sizes)
    )
    .withColumn(
        "array_length_mismatch",
        when(
            col("min_array_length").isNull()
            | col("max_array_length").isNull(),
            lit(False)
        ).otherwise(
            col("min_array_length")
            != col("max_array_length")
        )
    )
)

print("Null-safe array validation created.")
print("Missing arrays will no longer be treated as length -1.")
print("Actual unequal array lengths will still be detected.")

print("\nSECTION 3 COMPLETED")


FOR1SIGHT — NULL-SAFE ARRAY VALIDATION
Total global array fields: 43
Null-safe array validation created.
Missing arrays will no longer be treated as length -1.
Actual unequal array lengths will still be detected.

SECTION 3 COMPLETED


## 4. Flatten arrays into individual lap records

`arrays_zip()` aligns each field by array index. `explode()` then turns each zipped element into one lap-level row.

Metadata is extracted from the folder structure:

`raw_hf_data / year / event / session / driver / laptimes.json`

The original lap fields are retained.


In [16]:
# Use the original schema here so the three audit columns added above
# are selected only once in the following projection.
scalar_cols = [
    field.name for field in lap_df.schema.fields
    if not isinstance(field.dataType, ArrayType)
]

# Treat absent optional arrays as empty arrays for arrays_zip().
zip_inputs = []
for field in array_fields:
    empty_array = array_repeat(
        lit(None).cast(field.dataType.elementType),
        0
    )
    zip_inputs.append(
        coalesce(col(field.name), empty_array).alias(field.name)
    )

exploded_df = (
    lap_with_lengths
    .select(
        *[col(name) for name in scalar_cols],
        "min_array_length",
        "max_array_length",
        "array_length_mismatch",
        explode(arrays_zip(*zip_inputs)).alias("_lap_record")
    )
)

flat_df = exploded_df.select(
    *[col(name) for name in scalar_cols],
    "min_array_length",
    "max_array_length",
    "array_length_mismatch",
    *[
        col("_lap_record").getField(field.name).alias(field.name)
        for field in array_fields
    ]
)

# Normalize path separators before extracting folder metadata.
flat_df = (
    flat_df
    .withColumn(
        "_normalized_path",
        regexp_replace(col("source_file"), r"\\", "/")
    )
    .withColumn(
        "year",
        regexp_extract(
            col("_normalized_path"),
            r"(?i)(?:^|/)raw_hf_data/([^/]+)/",
            1
        ).cast("int")
    )
    .withColumn(
        "event_name",
        regexp_extract(
            col("_normalized_path"),
            r"(?i)(?:^|/)raw_hf_data/[^/]+/([^/]+)/",
            1
        )
    )
    .withColumn(
        "session_type",
        regexp_extract(
            col("_normalized_path"),
            r"(?i)(?:^|/)raw_hf_data/[^/]+/[^/]+/([^/]+)/",
            1
        )
    )
    .withColumn(
        "driver_folder",
        regexp_extract(
            col("_normalized_path"),
            r"(?i)(?:^|/)raw_hf_data/[^/]+/[^/]+/[^/]+/([^/]+)/laptimes\.json$",
            1
        )
    )
)

drv_text = lower(trim(col("drv").cast("string")))
driver_from_array = when(
    drv_text.isNull() | drv_text.isin("", "none", "null", "nan"),
    lit(None)
).otherwise(upper(trim(col("drv").cast("string"))))

flat_df = (
    flat_df
    .withColumn(
        "driver_code",
        coalesce(driver_from_array, upper(trim(col("driver_folder"))))
    )
    .withColumn("lap", col("lap").cast("int"))
    .withColumn("time", col("time").cast("double"))
)

# Apply event/session decoding now so the staging dataset is already standardised.
flat_df = (
    flat_df
    .withColumn("event_name", trim(regexp_replace(col("event_name"), "%20", " ")))
    .withColumn("session_type", trim(regexp_replace(col("session_type"), "%20", " ")))
    .withColumn("driver_code", upper(trim(col("driver_code"))))
)

print("Lap-level transformation plan created.")
print("No count, show, join or dropDuplicates action has been executed.")


Lap-level transformation plan created.
No count, show, join or dropDuplicates action has been executed.


## 5. Apply row-level quality flags

Correct source flag meanings:

| Field | Meaning | Exclude when |
|---|---|---|
| `del` | Lap deleted | `true` |
| `iacc` | Lap timing is accurate | `false` |

Unknown/null flag values are not automatically rejected. The missing-critical rule excludes rows without lap number, lap time or driver code.


In [17]:
deleted_flag = coalesce(
    lower(trim(col("del").cast("string"))).isin("true", "1", "yes"),
    lit(False)
)

inaccurate_flag = coalesce(
    lower(trim(col("iacc").cast("string"))).isin("false", "0", "no"),
    lit(False)
)

missing_critical_expr = (
    col("lap").isNull()
    | col("time").isNull()
    | col("driver_code").isNull()
)

flagged_df = (
    flat_df
    .withColumn("is_deleted", deleted_flag)
    .withColumn("is_inaccurate", inaccurate_flag)
    .withColumn("missing_critical", missing_critical_expr)
)

clean_condition = (
    (~col("is_deleted"))
    & (~col("is_inaccurate"))
    & (~col("missing_critical"))
)

print("Row-level flags and clean-record condition created.")


Row-level flags and clean-record condition created.


## 6. Materialise one temporary Parquet staging dataset

This is the key performance change. The first full Spark action writes the flattened and flagged lap records to a temporary Parquet directory under Windows TEMP (no spaces in the path).

After this write completes, all quality checks read the staged Parquet files. They no longer need to re-run the 6,904-file JSON transformation lineage.

The staging directory is temporary working data, not the final Silver warehouse.


In [18]:
# Only the temporary staging directory is replaced here.
if STAGING_PATH.exists():
    shutil.rmtree(STAGING_PATH)

print("Writing temporary flattened-lap staging data...")
print("This is the main raw-JSON processing job. It may take time, but it should run once.")

(
    flagged_df
    .coalesce(8)
    .write
    .mode("overwrite")
    .parquet(STAGING_URI)
)

print("STAGING WRITE COMPLETED.")

# Read back the staged Parquet. Later checks use this, not the raw JSON.
staged_df = spark.read.parquet(STAGING_URI)

print("Staging Parquet is ready for validation.")
staged_df.printSchema()


Writing temporary flattened-lap staging data...
This is the main raw-JSON processing job. It may take time, but it should run once.
STAGING WRITE COMPLETED.
Staging Parquet is ready for validation.
root
 |-- source_file: string (nullable = true)
 |-- min_array_length: integer (nullable = true)
 |-- max_array_length: integer (nullable = true)
 |-- array_length_mismatch: boolean (nullable = true)
 |-- compound: string (nullable = true)
 |-- dNum: string (nullable = true)
 |-- del: string (nullable = true)
 |-- delR: string (nullable = true)
 |-- drv: string (nullable = true)
 |-- ff1G: boolean (nullable = true)
 |-- fresh: boolean (nullable = true)
 |-- iacc: boolean (nullable = true)
 |-- lSD: string (nullable = true)
 |-- lST: string (nullable = true)
 |-- lap: integer (nullable = true)
 |-- life: string (nullable = true)
 |-- ms1: string (nullable = true)
 |-- ms2: string (nullable = true)
 |-- ms3: string (nullable = true)
 |-- pb: string (nullable = true)
 |-- pin: string (nullable 

## 7. Consolidated data-quality validation

One aggregate computes the main record counts in a single pass over the staging Parquet. Exclusion counts can overlap: a row may be both inaccurate and missing a critical field.

We deliberately do not call `dropDuplicates()` across all columns. Instead, duplicate analytical keys are checked separately and reported, not silently removed.


In [19]:
summary_row = staged_df.agg(
    count("*").alias("flattened_rows"),
    F_sum(when(col("is_deleted"), 1).otherwise(0)).alias("deleted_rows"),
    F_sum(when(col("is_inaccurate"), 1).otherwise(0)).alias("inaccurate_rows"),
    F_sum(when(col("missing_critical"), 1).otherwise(0)).alias("missing_critical_rows"),
    F_sum(when(clean_condition, 1).otherwise(0)).alias("clean_rows"),
    countDistinct(
        when(col("array_length_mismatch"), col("source_file"))
    ).alias("array_mismatch_files"),
    F_sum(
        when(
            clean_condition
            & (
                col("year").isNull()
                | col("event_name").isNull()
                | (trim(col("event_name")) == "")
                | col("session_type").isNull()
                | (trim(col("session_type")) == "")
                | col("driver_code").isNull()
                | (trim(col("driver_code")) == "")
            ),
            1
        ).otherwise(0)
    ).alias("clean_rows_missing_metadata")
).first().asDict()

flattened_rows = int(summary_row["flattened_rows"] or 0)
deleted_rows = int(summary_row["deleted_rows"] or 0)
inaccurate_rows = int(summary_row["inaccurate_rows"] or 0)
missing_critical_rows = int(summary_row["missing_critical_rows"] or 0)
clean_rows = int(summary_row["clean_rows"] or 0)
mismatch_file_count = int(summary_row["array_mismatch_files"] or 0)
metadata_missing_count = int(summary_row["clean_rows_missing_metadata"] or 0)

clean_staged_df = staged_df.filter(clean_condition)

key_columns = ["year", "event_name", "session_type", "driver_code", "lap"]

duplicate_keys_df = (
    clean_staged_df
    .groupBy(*key_columns)
    .count()
    .filter(col("count") > 1)
)

duplicate_key_groups = duplicate_keys_df.count()

print("=" * 72)
print("FOR1SIGHT V2 — DATA QUALITY SUMMARY")
print("=" * 72)
print(f"Source JSON files:                  {len(laptimes_files):,}")
print(f"Flattened lap records:              {flattened_rows:,}")
print(f"Deleted lap rows (overlapping):     {deleted_rows:,}")
print(f"Inaccurate lap rows (overlapping):  {inaccurate_rows:,}")
print(f"Missing critical fields:            {missing_critical_rows:,}")
print(f"Clean lap records:                  {clean_rows:,}")
print(f"Array-mismatch source files:        {mismatch_file_count:,}")
print(f"Clean rows missing metadata:        {metadata_missing_count:,}")
print(f"Duplicate lap-key groups:           {duplicate_key_groups:,}")
print("Note: exclusion counts overlap and must not be added together.")

print("\nDuplicate lap keys, if any:")
duplicate_keys_df.show(20, truncate=False)


FOR1SIGHT V2 — DATA QUALITY SUMMARY
Source JSON files:                  6,904
Flattened lap records:              197,413
Deleted lap rows (overlapping):     1,974
Inaccurate lap rows (overlapping):  61,180
Missing critical fields:            28,261
Clean lap records:                  134,356
Array-mismatch source files:        0
Clean rows missing metadata:        0
Duplicate lap-key groups:           0
Note: exclusion counts overlap and must not be added together.

Duplicate lap keys, if any:
+----+----------+------------+-----------+---+-----+
|year|event_name|session_type|driver_code|lap|count|
+----+----------+------------+-----------+---+-----+
+----+----------+------------+-----------+---+-----+



In [ ]:

# ==========================================================
# SECTION 7.1 — ARRAY MISMATCH INVESTIGATION (FIXED)
# ==========================================================

from urllib.parse import urlparse, unquote
from pathlib import Path

from pyspark.sql.functions import (
    col, size, when, lit, input_file_name
)
from pyspark.sql.types import ArrayType

print("=" * 70)
print("ARRAY LENGTH MISMATCH INVESTIGATION — FIXED")
print("=" * 70)


# ----------------------------------------------------------
# 1. CONVERT SPARK FILE URI TO WINDOWS PATH
# ----------------------------------------------------------

def spark_uri_to_local_path(uri):

    parsed = urlparse(uri)
    decoded_path = unquote(parsed.path)

    # Convert /C:/Users/... into C:/Users/...
    if (
        len(decoded_path) >= 3
        and decoded_path[0] == "/"
        and decoded_path[2] == ":"
    ):
        decoded_path = decoded_path[1:]

    return str(Path(decoded_path))


# ----------------------------------------------------------
# 2. SELECT FIRST 10 MISMATCHED SOURCE FILES
# ----------------------------------------------------------

mismatch_uri_rows = (
    mismatch_sources
    .select("source_file")
    .distinct()
    .orderBy("source_file")
    .limit(10)
    .collect()
)

mismatch_local_paths = [
    spark_uri_to_local_path(row["source_file"])
    for row in mismatch_uri_rows
]

existing_paths = [
    p for p in mismatch_local_paths
    if Path(p).exists()
]

missing_paths = [
    p for p in mismatch_local_paths
    if not Path(p).exists()
]

print("Files selected:", len(mismatch_local_paths))
print("Files found:", len(existing_paths))
print("Files not found:", len(missing_paths))

if missing_paths:
    print("\nMISSING PATHS:")
    for p in missing_paths:
        print(p)


# ----------------------------------------------------------
# 3. READ ORIGINAL JSON FILES
# ----------------------------------------------------------

if existing_paths:

    mismatch_raw = (
        spark.read
        .option("multiLine", "true")
        .json(existing_paths)
    )

    # Only inspect array fields available in these files.
    sample_array_names = [
        field.name
        for field in mismatch_raw.schema.fields
        if isinstance(field.dataType, ArrayType)
    ]

    print("\nArray fields found:", len(sample_array_names))

    # ------------------------------------------------------
    # 4. CALCULATE ARRAY LENGTHS
    # ------------------------------------------------------

    length_expressions = [
        when(
            col(name).isNotNull(),
            size(col(name))
        )
        .otherwise(lit(None).cast("int"))
        .alias(name + "_length")

        for name in sample_array_names
    ]

    length_df = mismatch_raw.select(
        input_file_name().alias("_source_uri"),
        *length_expressions
    )

    # ------------------------------------------------------
    # 5. IDENTIFY ACTUAL LENGTH DIFFERENCES
    # ------------------------------------------------------

    print("\nINDIVIDUAL FILE RESULTS")

    for row in length_df.collect():

        values = row.asDict()

        local_file = spark_uri_to_local_path(
            values["_source_uri"]
        )

        lengths = {
            name: values[name + "_length"]
            for name in sample_array_names
            if values[name + "_length"] is not None
        }

        unique_lengths = sorted(set(lengths.values()))

        print("\n" + "-" * 65)
        print("FILE:", local_file)

        print("Present arrays:", len(lengths))
        print("Different lengths:", unique_lengths)

        if len(unique_lengths) > 1:

            print("RESULT: ACTUAL ARRAY-LENGTH DIFFERENCE")

            for name, length in lengths.items():
                print(f"{name:15} : {length}")

        else:

            print("RESULT: NO DIFFERENCE AMONG PRESENT ARRAYS")

        missing_arrays = [
            name for name in sample_array_names
            if values[name + "_length"] is None
        ]

        print("Missing arrays:", missing_arrays)

else:

    print("\nNo accessible source files to inspect.")

print("\nARRAY INVESTIGATION COMPLETED")


## 8. Season, event and session coverage

These checks now read the staged Parquet dataset. They do not reprocess the raw JSON files.


In [21]:
print("=" * 72)
print("CLEAN RECORDS BY SEASON")
print("=" * 72)

(
    clean_staged_df
    .groupBy("year")
    .agg(
        count("*").alias("clean_lap_records"),
        countDistinct("event_name").alias("events_present"),
        countDistinct("driver_code").alias("drivers_present")
    )
    .orderBy("year")
    .show(20, truncate=False)
)

print("\nCLEAN RECORDS BY SEASON AND SESSION")

(
    clean_staged_df
    .groupBy("year", "session_type")
    .count()
    .orderBy("year", "session_type")
    .show(100, truncate=False)
)

print("\nSAMPLE EVENT NAMES")

(
    clean_staged_df
    .select("year", "event_name")
    .distinct()
    .orderBy("year", "event_name")
    .show(80, truncate=False)
)


CLEAN RECORDS BY SEASON
+----+-----------------+--------------+---------------+
|year|clean_lap_records|events_present|drivers_present|
+----+-----------------+--------------+---------------+
|2024|47673            |25            |33             |
|2025|46811            |25            |40             |
|2026|39872            |20            |32             |
+----+-----------------+--------------+---------------+


CLEAN RECORDS BY SEASON AND SESSION
+----+-----------------+-----+
|year|session_type     |count|
+----+-----------------+-----+
|2024|Practice 1       |7598 |
|2024|Practice 2       |6785 |
|2024|Practice 3       |4241 |
|2024|Qualifying       |3014 |
|2024|Race             |23256|
|2024|Sprint           |2163 |
|2024|Sprint Qualifying|616  |
|2025|Practice 1       |8240 |
|2025|Practice 2       |6422 |
|2025|Practice 3       |4399 |
|2025|Qualifying       |2699 |
|2025|Race             |22760|
|2025|Sprint           |1650 |
|2025|Sprint Qualifying|641  |
|2026|Practice 1   

## 9. Regression test — SAI, Singapore Grand Prix, Race, Lap 26

This checks that the valid lap previously investigated remains in the clean dataset and that its sector times sum to its lap time.


In [22]:
lap26_check = (
    clean_staged_df
    .filter(
        (col("year") == 2026)
        & lower(col("event_name")).contains("singapore")
        & (lower(trim(col("session_type"))) == "race")
        & (upper(trim(col("driver_code"))) == "SAI")
        & (col("lap") == 26)
    )
    .withColumn(
        "sector_sum",
        col("s1").cast("double")
        + col("s2").cast("double")
        + col("s3").cast("double")
    )
    .withColumn(
        "sector_time_difference",
        col("time").cast("double") - col("sector_sum")
    )
)

lap26_count = lap26_check.count()
print("Matching clean SAI Singapore Race Lap 26 rows:", lap26_count)

lap26_check.select(
    "year", "event_name", "session_type", "driver_code",
    "lap", "time", "s1", "s2", "s3",
    "sector_sum", "sector_time_difference", "del", "iacc"
).show(truncate=False)


Matching clean SAI Singapore Race Lap 26 rows: 1
+----+--------------------+------------+-----------+---+-------+------+------+------+----------+----------------------+-----+----+
|year|event_name          |session_type|driver_code|lap|time   |s1    |s2    |s3    |sector_sum|sector_time_difference|del  |iacc|
+----+--------------------+------------+-----------+---+-------+------+------+------+----------+----------------------+-----+----+
|2026|Singapore Grand Prix|Race        |SAI        |26 |100.106|29.676|42.413|28.017|100.106   |0.0                   |false|true|
+----+--------------------+------------+-----------+---+-------+------+------+------+----------+----------------------+-----+----+



## 10. Prepare the final dashboard-compatible warehouse DataFrame

The clean records are retained with the original lap-time columns. Temporary ETL flags and path-parsing helper columns are removed. `source_file` is retained for lineage.

Sector times are cast to `double`. Other fields are left in their source-inferred types until their values have been separately audited.


In [23]:
helper_columns = [
    "_normalized_path",
    "driver_folder",
    "min_array_length",
    "max_array_length",
    "array_length_mismatch",
    "is_deleted",
    "is_inaccurate",
    "missing_critical"
]

warehouse_df = clean_staged_df.drop(
    *[name for name in helper_columns if name in clean_staged_df.columns]
)

for sector in ["s1", "s2", "s3"]:
    if sector in warehouse_df.columns:
        warehouse_df = warehouse_df.withColumn(
            sector,
            col(sector).cast("double")
        )

print("=" * 72)
print("FINAL WAREHOUSE SCHEMA")
print("=" * 72)
warehouse_df.printSchema()

print("\nSector data types:")
for sector in ["s1", "s2", "s3"]:
    if sector in warehouse_df.columns:
        print(sector, warehouse_df.schema[sector].dataType)

print("\nTotal columns:", len(warehouse_df.columns))
print("Expected clean rows from validation:", f"{clean_rows:,}")
print("No final local or HDFS warehouse has been written yet.")


FINAL WAREHOUSE SCHEMA
root
 |-- source_file: string (nullable = true)
 |-- compound: string (nullable = true)
 |-- dNum: string (nullable = true)
 |-- del: string (nullable = true)
 |-- delR: string (nullable = true)
 |-- drv: string (nullable = true)
 |-- ff1G: boolean (nullable = true)
 |-- fresh: boolean (nullable = true)
 |-- iacc: boolean (nullable = true)
 |-- lSD: string (nullable = true)
 |-- lST: string (nullable = true)
 |-- lap: integer (nullable = true)
 |-- life: string (nullable = true)
 |-- ms1: string (nullable = true)
 |-- ms2: string (nullable = true)
 |-- ms3: string (nullable = true)
 |-- pb: string (nullable = true)
 |-- pin: string (nullable = true)
 |-- pos: string (nullable = true)
 |-- pout: string (nullable = true)
 |-- qs: string (nullable = true)
 |-- s1: double (nullable = true)
 |-- s1T: string (nullable = true)
 |-- s2: double (nullable = true)
 |-- s2T: string (nullable = true)
 |-- s3: double (nullable = true)
 |-- s3T: string (nullable = true)
 |-- se

## 11. Optional export — disabled by default

The final destinations are deliberately new paths. The existing `hf_laptimes.parquet` and the previously created HDFS Silver `hf_laptimes` directory are not overwritten.

- Local output: `for1sight_warehouse/hf_laptimes_clean.parquet`
- HDFS output: `hdfs://localhost:9000/for1sight_warehouse/silver/hf_laptimes_clean`

**Keep both switches False until the validation output has been reviewed.**


In [24]:

# ==========================================================
# SECTION 11 — LOCAL SILVER PARQUET EXPORT
# ==========================================================

import shutil

print("=" * 70)
print("FOR1SIGHT — LOCAL SILVER EXPORT")
print("=" * 70)

# 1. EXPORT SETTINGS
WRITE_LOCAL_PARQUET = True
WRITE_HDFS = False

# 2. FINAL DESTINATION
LOCAL_OUTPUT_PATH = (
    PROJECT_ROOT
    / "for1sight_warehouse"
    / "hf_laptimes_clean.parquet"
)

# 3. SAFETY VALIDATION
if mismatch_file_count > 0:
    raise RuntimeError("Array mismatch detected.")

if metadata_missing_count > 0:
    raise RuntimeError("Missing metadata detected.")

if duplicate_key_groups > 0:
    raise RuntimeError("Duplicate lap keys detected.")

if not WRITE_LOCAL_PARQUET:
    print("Local export disabled.")

else:

    # Never overwrite an existing warehouse.
    if LOCAL_OUTPUT_PATH.exists():
        raise FileExistsError(
            f"Destination already exists: {LOCAL_OUTPUT_PATH}"
        )

    # 4. PREPARE TEMPORARY DESTINATION
    if LOCAL_TEMP_EXPORT_PATH.exists():
        shutil.rmtree(LOCAL_TEMP_EXPORT_PATH)

    print("\nWriting Parquet to temporary location...")
    print("Destination:", LOCAL_TEMP_EXPORT_URI)

    (
        warehouse_df
        .coalesce(8)
        .write
        .mode("errorifexists")
        .parquet(LOCAL_TEMP_EXPORT_URI)
    )

    print("\nTemporary Parquet write completed.")

    # 5. COPY TO PROJECT WAREHOUSE
    LOCAL_OUTPUT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copytree(
        LOCAL_TEMP_EXPORT_PATH,
        LOCAL_OUTPUT_PATH
    )

    print("\nLOCAL SILVER EXPORT SUCCESSFUL")
    print("Final destination:", LOCAL_OUTPUT_PATH)

    print("\nExpected records:", f"{clean_rows:,}")
    print("HDFS export:", "DISABLED")

    print("\n" + "=" * 70)
    print("SECTION 11 COMPLETED")
    print("=" * 70)


FOR1SIGHT — LOCAL SILVER EXPORT

Writing Parquet to temporary location...
Destination: file:///C:/Users/ayepd/AppData/Local/Temp/for1sight_etl_v2/hf_laptimes_clean_export

Temporary Parquet write completed.

LOCAL SILVER EXPORT SUCCESSFUL
Final destination: C:\Users\ayepd\Desktop\For1Sight Project\for1sight_warehouse\hf_laptimes_clean.parquet

Expected records: 134,356
HDFS export: DISABLED

SECTION 11 COMPLETED


## 12. Optional read-back validation

Run this cell only after enabling and completing the relevant export. The local directory is copied to a temporary no-space path before Spark reads it, matching the Windows path workaround used in the existing For1Sight workflow.


In [25]:

# ==========================================================
# SECTION 12 — LOCAL PARQUET READ-BACK VALIDATION
# ==========================================================

import shutil
from pathlib import Path

print("=" * 70)
print("FOR1SIGHT — LOCAL PARQUET READ-BACK VALIDATION")
print("=" * 70)

# 1. Check exported directory
if not LOCAL_OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"Parquet output not found: {LOCAL_OUTPUT_PATH}"
    )

print("Local Parquet directory exists: YES")

# 2. Inspect Parquet files
parquet_files = list(
    LOCAL_OUTPUT_PATH.rglob("*.parquet")
)

print("Parquet part files:", len(parquet_files))

if not parquet_files:
    raise RuntimeError("No Parquet part files found.")

total_size = sum(
    f.stat().st_size for f in parquet_files
)

print(
    "Total Parquet size (MB):",
    round(total_size / (1024 * 1024), 2)
)

# 3. Copy to temporary path without spaces
VERIFY_PATH = TEMP_ROOT / "local_readback_verify"

if VERIFY_PATH.exists():
    shutil.rmtree(VERIFY_PATH)

shutil.copytree(
    LOCAL_OUTPUT_PATH,
    VERIFY_PATH
)

print("\nTemporary verification copy created.")

# 4. Read Parquet using Spark
readback_df = spark.read.parquet(
    VERIFY_PATH.as_uri()
)

print("Parquet successfully loaded into Spark.")

# 5. Validate actual row count
actual_rows = readback_df.count()
expected_rows = int(clean_rows)

print("\nROW COUNT VALIDATION")
print("Expected rows:", f"{expected_rows:,}")
print("Actual rows:", f"{actual_rows:,}")

row_count_match = actual_rows == expected_rows

print("Row count matches:", row_count_match)

# 6. Validate schema
expected_schema = [
    (f.name, f.dataType.simpleString())
    for f in warehouse_df.schema.fields
]

actual_schema = [
    (f.name, f.dataType.simpleString())
    for f in readback_df.schema.fields
]

schema_match = expected_schema == actual_schema

print("\nSCHEMA VALIDATION")
print("Expected columns:", len(expected_schema))
print("Actual columns:", len(actual_schema))
print("Schema matches:", schema_match)

if not schema_match:
    print("\nEXPECTED SCHEMA:")
    print(expected_schema)

    print("\nACTUAL SCHEMA:")
    print(actual_schema)

# 7. Verify existing warehouse still exists
OLD_WAREHOUSE = (
    PROJECT_ROOT
    / "for1sight_warehouse"
    / "hf_laptimes.parquet"
)

print("\nEXISTING WAREHOUSE")
print("Original dataset exists:", OLD_WAREHOUSE.exists())

# 8. Final validation result
print("\n" + "=" * 70)

if row_count_match and schema_match:

    print("LOCAL SILVER PARQUET VALIDATION PASSED")

else:

    print("LOCAL SILVER PARQUET VALIDATION FAILED")

print("=" * 70)

assert row_count_match, "Row count validation failed."
assert schema_match, "Schema validation failed."


FOR1SIGHT — LOCAL PARQUET READ-BACK VALIDATION
Local Parquet directory exists: YES
Parquet part files: 8
Total Parquet size (MB): 10.95

Temporary verification copy created.
Parquet successfully loaded into Spark.

ROW COUNT VALIDATION
Expected rows: 134,356
Actual rows: 134,356
Row count matches: True

SCHEMA VALIDATION
Expected columns: 48
Actual columns: 48
Schema matches: True

EXISTING WAREHOUSE
Original dataset exists: True

LOCAL SILVER PARQUET VALIDATION PASSED


## 13. Finish

Keep the original ETL notebook and raw JSON files unchanged. Once the new local Parquet and HDFS Silver datasets have passed read-back validation, test the existing Streamlit pages against the new dataset in a separate controlled change.

When finished, Spark can be stopped with the optional command below.


In [26]:
# Gracefully stop the Spark session

spark.stop()

print("For1Sight Spark session stopped successfully.")

For1Sight Spark session stopped successfully.
